In [ ]:
pip install "gymnasium[classic-control]" stable-baselines3 sb3-contrib pyyaml pydantic

In [ ]:
pip uninstall -y torch torchvision torchaudio torchdata torchtext dgl

In [ ]:
pip install torch==2.1.2 

In [ ]:
pip install torchdata==0.7.1

In [ ]:
pip install dgl -f https://data.dgl.ai/wheels/torch-2.1/repo.html

In [ ]:
!pip install "numpy<2.0.0" "pandas<2.2.2"

In [ ]:
pip install holidays

In [1]:
import datetime
import json
import numpy as np
import pandas as pd
import gymnasium as gym
from pathlib import Path

DEMAND = 0
CAPACITY = 1

def _build_special_days(year, n_days):
    import holidays as hl
    pt_holidays = hl.country_holidays("PT", years=[year])
    start = datetime.date(year, 1, 1)
    special = set()
    for d in range(n_days):
        date = start + datetime.timedelta(days=d)
        if date.weekday() == 6 or date in pt_holidays:
            special.add(d)
    return special


class ScheduleEnv(gym.Env):
    def __init__(self,
                 data_dir: str = "/kaggle/input/datasets/monteiroo21/4teams",
                 capacity_slack: int = 1):
        super().__init__()
        base = Path(data_dir)

        with open(base / "problem.json") as f:
            prob = json.load(f)

        self.num_days = prob["temporalScope"]["numDays"]
        self.year = prob["temporalScope"]["year"]
        employees = prob["employees"]["simple"]
        self.num_employees = len(employees)
        self.employee_teams = [set(emp.get("teams", [])) for emp in employees]
        self.dual_team = [len(teams) > 1 for teams in self.employee_teams]

        shifts_sorted = sorted(prob["demand"]["shifts"], key=lambda s: s["order"])
        self.shift_codes = [s["code"] for s in shifts_sorted]
        self.shift_idx = {c: i for i, c in enumerate(self.shift_codes)}
        self.shift_order_map = {s["code"]: s["order"] for s in shifts_sorted}
        self.teams = list(prob["demand"]["organizationalUnits"]["teams"])
        self.team_idx = {t: i for i, t in enumerate(self.teams)}
        self.num_shifts = len(self.shift_codes)
        self.num_teams = len(self.teams)

        self.team_sizes = {
            team: sum(1 for s in self.employee_teams if team in s)
            for team in self.teams
        }

        vac_df = pd.read_csv(base / "vacations.csv", header=None)
        self.vac_mask = vac_df.iloc[:, 1:].values.astype(bool)

        dem_df = pd.read_csv(base / "demand.csv")
        dem_df["date"] = pd.to_datetime(dem_df["date"])
        start_ts = pd.Timestamp(f"{self.year}-01-01")
        dem_df["day_idx"] = (dem_df["date"] - start_ts).dt.days

        self.min_demand = np.zeros((self.num_days, self.num_shifts, self.num_teams), dtype=int)
        self.ideal_demand = np.zeros((self.num_days, self.num_shifts, self.num_teams), dtype=int)
        for _, row in dem_df.iterrows():
            d = int(row["day_idx"])
            s = self.shift_idx[row["shift"]]
            t = self.team_idx[row["team"]]
            self.min_demand[d, s, t] = int(row["minimum"])
            self.ideal_demand[d, s, t] = int(row["ideal"])

        self.special_days = _build_special_days(self.year, self.num_days)

        self.max_days_per_year = 223
        self.max_consecutive_days = 5
        self.special_days_cap = 22
        self.capacity_slack = capacity_slack

        # Lower bound on achievable ideal shortfall: total ideal headcount can
        # exceed the workforce's total workable employee-days, so judge the
        # ideal gap as excess over this floor.
        self.ideal_floor = int(max(0, self.ideal_demand.sum()
                                   - self.num_employees * self.max_days_per_year))

        self.PASS_ACTION = self.num_employees
        self.action_space = gym.spaces.Discrete(self.num_employees + 1)
        self.observation_space = gym.spaces.Box(low=0.0, high=1.0, shape=(1,), dtype=np.float32)

        self.reset()

    def build_slot_queue(self, min_demand, capacity_slack=1):
        num_days, num_shifts, num_teams = min_demand.shape
        demand_slots = []
        for d in range(num_days):
            for s in range(num_shifts):
                for t in range(num_teams):
                    headcount = int(min_demand[d, s, t])
                    for _ in range(headcount):
                        demand_slots.append((d, s, t, DEMAND))

        capacity_slots = []
        for d in range(num_days):
            for s in range(num_shifts):
                for t in range(num_teams):
                    deficit = int(self.ideal_demand[d, s, t] - min_demand[d, s, t])
                    for _ in range(deficit):
                        capacity_slots.append((d, s, t, CAPACITY))

        for _ in range(capacity_slack):
            for d in range(num_days):
                for s in range(num_shifts):
                    for t in range(num_teams):
                        capacity_slots.append((d, s, t, CAPACITY))

        return demand_slots + capacity_slots, len(demand_slots)

    def _get_assigned_shift(self, emp, day):
        s = self.emp_day_shift[emp, day]
        if s < 0:
            return None
        return self.shift_codes[s]

    def _get_prev_shift(self, emp, day):
        if day == 0:
            return None
        return self._get_assigned_shift(emp, day - 1)

    def _get_next_shift(self, emp, day):
        if day >= self.num_days - 1:
            return None
        return self._get_assigned_shift(emp, day + 1)

    def _consecutive_streak_if_work(self, emp, day):
        streak = 1
        d = day - 1
        while d >= 0 and self.emp_day_shift[emp, d] >= 0:
            streak += 1
            d -= 1
        d = day + 1
        while d < self.num_days and self.emp_day_shift[emp, d] >= 0:
            streak += 1
            d += 1
        return streak

    def _get_info(self):
        return {}

    def _obs(self):
        return np.zeros(1, dtype=np.float32)

    def current_slot(self):
        if self.slot_idx >= len(self.slot_queue):
            return None
        return self.slot_queue[self.slot_idx]

    def current_day(self):
        s = self.current_slot()
        return s[0] if s is not None else 0

    def _team_balance_bonus(self, emp_id, team):
        if not self.dual_team[emp_id]:
            return 0.0
        sizes = {t: self.team_sizes[t] for t in self.employee_teams[emp_id]}
        min_size, max_size = min(sizes.values()), max(sizes.values())
        if min_size == max_size:
            return 0.0
        smaller = {t for t, s in sizes.items() if s == min_size}
        return (max_size - min_size) * 0.5 if team in smaller else 0.0

    def _calculate_reward(self, action, day, s_idx, t_idx, kind):
        if action == self.PASS_ACTION:
            return -5.0 if kind == DEMAND else 0.0
        team = self.teams[t_idx]
        bonus = self._team_balance_bonus(action, team)
        if kind == DEMAND:
            return 1.0 + 0.2 * bonus
        cov = self.daily_coverage[day, s_idx, t_idx]
        if cov <= self.ideal_demand[day, s_idx, t_idx]:
            return 1.0 + 0.1 * bonus
        return 0.2

    def _calculate_final_reward(self):
        shortfall = int(np.maximum(0, self.min_demand - self.daily_coverage).sum())
        self.ideal_shortfall = int(np.maximum(0, self.ideal_demand - self.daily_coverage).sum())
        days_short = int(np.maximum(0, self.max_days_per_year - self.days_worked).sum())
        total_min = self.min_demand.sum()
        ideal_excess = max(0, self.ideal_shortfall - self.ideal_floor)
        reward = 200.0 * (1.0 - shortfall / total_min)
        reward -= 1000.0 * ideal_excess / total_min
        reward -= 10000.0 * days_short / (self.num_employees * self.max_days_per_year)
        return reward

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.days_worked = np.zeros(self.num_employees, dtype=int)
        self.special_days_worked = np.zeros(self.num_employees, dtype=int)
        self.daily_coverage = np.zeros(
            (self.num_days, self.num_shifts, self.num_teams), dtype=np.float32
        )
        self.emp_day_shift = np.full((self.num_employees, self.num_days), -1, dtype=int)
        self.emp_day_team = np.full((self.num_employees, self.num_days), -1, dtype=int)
        self.slot_queue, self.num_demand_slots = self.build_slot_queue(self.min_demand, self.capacity_slack)
        self.slot_idx = 0
        self.slot_filled = np.zeros(len(self.slot_queue), dtype=bool)
        self.demand_skips = 0
        self.ideal_shortfall = 0
        return self._obs(), self._get_info()

    def step(self, action):
        if self.slot_idx >= len(self.slot_queue):
            return self._obs(), 0.0, True, False, self._get_info()
        day, s_idx, t_idx, kind = self.slot_queue[self.slot_idx]
        action = int(action)

        if action != self.PASS_ACTION:
            self.slot_filled[self.slot_idx] = True
            self.emp_day_shift[action, day] = s_idx
            self.emp_day_team[action, day] = t_idx
            self.days_worked[action] += 1
            self.daily_coverage[day, s_idx, t_idx] += 1
            if day in self.special_days:
                self.special_days_worked[action] += 1
        elif kind == DEMAND:
            self.demand_skips += 1

        reward = self._calculate_reward(action, day, s_idx, t_idx, kind)
        self.slot_idx += 1
        terminated = self.slot_idx >= len(self.slot_queue)
        if terminated:
            reward += self._calculate_final_reward()
        return self._obs(), reward, terminated, False, self._get_info()

    def _emp_can_cover(self, emp_id, day_id, shift_id, team_id):
        if self.teams[team_id] not in self.employee_teams[emp_id]:
            return False
        if self.vac_mask[emp_id, day_id]:
            return False
        if self.emp_day_shift[emp_id, day_id] >= 0:
            return False
        if self.days_worked[emp_id] >= self.max_days_per_year:
            return False
        if day_id in self.special_days and self.special_days_worked[emp_id] >= self.special_days_cap:
            return False
        if self._consecutive_streak_if_work(emp_id, day_id) > self.max_consecutive_days:
            return False

        prev_shift = self._get_prev_shift(emp_id, day_id)
        next_shift = self._get_next_shift(emp_id, day_id)
        shift_code = self.shift_codes[shift_id]

        if shift_code == "M" and prev_shift == "T":
            return False
        if shift_code == "T" and next_shift == "M":
            return False

        return True

    def get_employee_mask(self):
        mask = np.zeros(self.num_employees + 1, dtype=bool)
        if self.slot_idx >= len(self.slot_queue):
            mask[self.PASS_ACTION] = True
            return mask
        day, s_idx, t_idx, kind = self.slot_queue[self.slot_idx]
        for e in range(self.num_employees):
            if self._emp_can_cover(e, day, s_idx, t_idx):
                mask[e] = True
        if kind == CAPACITY:
            mask[self.PASS_ACTION] = True
        elif not mask[:self.num_employees].any():
            mask[self.PASS_ACTION] = True
        return mask

    def action_label(self, emp, day):
        s = self.emp_day_shift[emp, day]
        if s < 0:
            return "-"
        t = self.emp_day_team[emp, day]
        return f"{self.shift_codes[s]}-{self.teams[t]}"

    def render(self):
        for emp in range(self.num_employees):
            schedule = [self.action_label(emp, day) for day in range(self.num_days)]
            print(f"Employee {emp + 1:2d}: {schedule}")

In [ ]:
import dgl
import dgl.nn as dglnn
import torch
import numpy as np

def emp_feat_dim(env):
    return 3


def demand_feat_dim(env):
    return env.num_shifts + 7


def team_feat_dim(env):
    return 1 + 2 * env.num_shifts


def _o7_static(env):
    if not hasattr(env, "_o7_cache"):
        slots = env.slot_queue
        ND = len(slots)
        S = env.num_shifts

        slot_day = np.array([d for (d, _, _, _) in slots], dtype=np.int64)
        slot_shift = np.array([s for (_, s, _, _) in slots], dtype=np.int64)
        slot_team = np.array([t for (_, _, t, _) in slots], dtype=np.int64)
        slot_kind = np.array([k for (_, _, _, k) in slots], dtype=np.float32)

        et_src, et_dst = [], []
        for emp in range(env.num_employees):
            for team in env.employee_teams[emp]:
                et_src.append(emp)
                et_dst.append(env.team_idx[team])

        # employee -> demand wherever qualified (team member, not on vacation)
        emp_in_team = np.zeros((env.num_employees, env.num_teams), dtype=bool)
        for emp in range(env.num_employees):
            for team in env.employee_teams[emp]:
                emp_in_team[emp, env.team_idx[team]] = True
        eligible = emp_in_team[:, slot_team] & ~env.vac_mask[:, slot_day]  # (E, ND)
        ed_src, ed_dst = np.where(eligible)

        # Static demand-feature columns; dynamic ones stay zero here.
        feats = np.zeros((ND, demand_feat_dim(env)), dtype=np.float32)
        feats[np.arange(ND), slot_shift] = 1.0
        feats[:, S] = slot_kind
        feats[:, S + 3] = slot_day / env.num_days
        feats[:, S + 4] = np.array([float(d in env.special_days) for d in slot_day.tolist()],
                                   dtype=np.float32)

        env._o7_cache = {
            "slot_day": slot_day, "slot_shift": slot_shift, "slot_team": slot_team,
            "slot_ids": np.arange(ND),
            "et": (torch.tensor(et_src), torch.tensor(et_dst)),
            "ed": (torch.from_numpy(ed_src), torch.from_numpy(ed_dst)),
            "dt": (torch.arange(ND), torch.from_numpy(slot_team)),
            "demand_static": feats,
        }
    return env._o7_cache


def _static_feats(env):
    if not hasattr(env, "_static_feat_cache"):
        emp_pos = (np.arange(env.num_employees) / env.num_employees).astype(np.float32)
        team_size = np.array([env.team_sizes[t] / env.num_employees for t in env.teams],
                             dtype=np.float32)
        env._static_feat_cache = (emp_pos, team_size)
    return env._static_feat_cache


def build_graph(env):
    c = _o7_static(env)
    et_src, et_dst = c["et"]
    ed_src, ed_dst = c["ed"]
    dt_src, dt_dst = c["dt"]
    return dgl.heterograph({
        ("employee", "member_of", "team"): (et_src, et_dst),
        ("team", "has_member", "employee"): (et_dst, et_src),
        ("demand", "belongs_to", "team"): (dt_src, dt_dst),
        ("team", "has_demand", "demand"): (dt_dst, dt_src),
        ("employee", "qualified", "demand"): (ed_src, ed_dst),
        ("demand", "qualified_by", "employee"): (ed_dst, ed_src),
    }, num_nodes_dict={
        "employee": env.num_employees,
        "demand": len(env.slot_queue),
        "team": env.num_teams,
    })


def get_graph(env):
    # Topology is fully static: one graph per scenario, features refreshed per
    # day via update_graph_features.
    if not hasattr(env, "_o7_graph"):
        env._o7_graph = build_graph(env)
    return env._o7_graph


def update_graph_features(g, env):
    c = _o7_static(env)
    emp_pos, team_size = _static_feats(env)
    S = env.num_shifts

    emp_feats = np.empty((env.num_employees, emp_feat_dim(env)), dtype=np.float32)
    emp_feats[:, 0] = env.days_worked / env.max_days_per_year
    cur_day = env.current_day()
    for emp in range(env.num_employees):
        emp_feats[emp, 1] = env._consecutive_streak_if_work(emp, cur_day) / env.max_consecutive_days
    emp_feats[:, 2] = emp_pos

    min_gap = env.min_demand - env.daily_coverage      # (D, S, T)
    ideal_gap = env.ideal_demand - env.daily_coverage  # (D, S, T)

    dem_feats = c["demand_static"].copy()
    dem_feats[:, S + 1] = env.slot_filled
    dem_feats[:, S + 2] = c["slot_ids"] < env.slot_idx
    dem_feats[:, S + 5] = min_gap[c["slot_day"], c["slot_shift"], c["slot_team"]] / 10.0
    dem_feats[:, S + 6] = ideal_gap[c["slot_day"], c["slot_shift"], c["slot_team"]] / 10.0

    team_feats = np.empty((env.num_teams, team_feat_dim(env)), dtype=np.float32)
    team_feats[:, 0] = team_size
    team_feats[:, 1:1 + S] = min_gap[cur_day].T / 10.0
    team_feats[:, 1 + S:1 + 2 * S] = ideal_gap[cur_day].T / 10.0

    emp_t = torch.from_numpy(emp_feats)
    dem_t = torch.from_numpy(dem_feats)
    team_t = torch.from_numpy(team_feats)
    g.nodes["employee"].data["feat"] = emp_t
    g.nodes["demand"].data["feat"] = dem_t
    g.nodes["team"].data["feat"] = team_t
    return {"employee": emp_t, "demand": dem_t, "team": team_t}


def slot_gap(env, day_id, s_idx, t_idx):
    cov = env.daily_coverage[day_id, s_idx, t_idx]
    return np.array([
        env.min_demand[day_id, s_idx, t_idx] - cov,
        env.ideal_demand[day_id, s_idx, t_idx] - cov,
    ], dtype=np.float32)


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class GNNActorCritic(nn.Module):
    def __init__(self, emp_in_feats, demand_in_feats, team_in_feats, num_shifts,
                 hidden_dim=64, encoded_dim=64):
        super().__init__()
        self.encoded_dim = encoded_dim
        self.num_shifts = num_shifts

        self.emp_proj = nn.Linear(emp_in_feats, hidden_dim)
        self.demand_proj = nn.Linear(demand_in_feats, hidden_dim)
        self.team_proj = nn.Linear(team_in_feats, hidden_dim)

        rel_names = ["member_of", "has_member", "belongs_to", "has_demand",
                     "qualified", "qualified_by"]

        self.conv1 = dglnn.HeteroGraphConv({
            rel: dglnn.SAGEConv(hidden_dim, hidden_dim, 'mean') for rel in rel_names
        }, aggregate='sum')

        self.conv2 = dglnn.HeteroGraphConv({
            rel: dglnn.SAGEConv(hidden_dim, encoded_dim, 'mean') for rel in rel_names
        }, aggregate='sum')

        # ctx: current demand-node emb + its team emb + raw slot gaps + shift one-hot + kind
        ctx_dim = encoded_dim + encoded_dim + 2 + self.num_shifts + 1  # 64+64+2+2+1 = 133
        head_in = encoded_dim + ctx_dim

        # Pointer score: one logit per employee
        self.score_head  = nn.Sequential(
            nn.Linear(head_in, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )

        # PASS + value condition on a workforce summary (mean emp_emb) + slot ctx.
        self.pass_head   = nn.Sequential(
            nn.Linear(head_in, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )

        self.critic_head = nn.Sequential(
            nn.Linear(head_in, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )

    @classmethod
    def from_env(cls, env, hidden_dim=64, encoded_dim=64):
        return cls(
            emp_in_feats=emp_feat_dim(env),
            demand_in_feats=demand_feat_dim(env),
            team_in_feats=team_feat_dim(env),
            num_shifts=env.num_shifts,
            hidden_dim=hidden_dim,
            encoded_dim=encoded_dim,
        )

    def gnn_forward(self, g):
        h = {
            "employee": self.emp_proj(g.nodes["employee"].data["feat"]),
            "demand": self.demand_proj(g.nodes["demand"].data["feat"]),
            "team": self.team_proj(g.nodes["team"].data["feat"]),
        }

        h = self.conv1(g, h)
        h = {k: F.relu(v) for k, v in h.items()}
        h = self.conv2(g, h)
        h = {k: F.relu(v) for k, v in h.items()}
        return h["employee"], h["demand"], h["team"]

    def _heads_multi(self, emp_emb_b, demand_vec, team_vec, slot_gaps, shift, kind, action_masks):
        B, E, _ = emp_emb_b.shape
        ctx = torch.cat([demand_vec, team_vec, slot_gaps / 10.0, shift, kind], dim=-1)  # (B, 133)

        ctx_b = ctx.unsqueeze(1).expand(B, E, -1)
        emp_logits = self.score_head(torch.cat([emp_emb_b, ctx_b], dim=-1)).squeeze(-1)  # (B, E)

        pooled = torch.cat([emp_emb_b.mean(1), ctx], dim=-1)
        pass_logit = self.pass_head(pooled)                                       # (B, 1)
        values = self.critic_head(pooled).squeeze(-1)                             # (B,)

        logits = torch.cat([emp_logits, pass_logit], dim=-1)                      # (B, E+1)
        masks_bool = torch.as_tensor(action_masks, dtype=torch.bool)
        if masks_bool.dim() == 1:
            masks_bool = masks_bool.unsqueeze(0)
        all_invalid = ~masks_bool.any(dim=-1)
        if all_invalid.any():
            masks_bool = masks_bool.clone()
            masks_bool[all_invalid, -1] = True
        logits = logits.masked_fill(~masks_bool, float("-inf"))
        return F.softmax(logits, dim=-1), values

    def _heads(self, emp_emb, demand_emb, team_emb, demand_ids, team_ids,
               slot_gaps, shift, kind, action_masks):
        # Single-snapshot case: the whole batch shares one embedding set.
        B = demand_ids.shape[0]
        emp_emb_b = emp_emb.unsqueeze(0).expand(B, -1, -1)
        return self._heads_multi(emp_emb_b, demand_emb[demand_ids], team_emb[team_ids],
                                 slot_gaps, shift, kind, action_masks)

    def forward(self, g, demand_ids, team_ids, slot_gaps, shift, kind, action_masks):
        emp_emb, demand_emb, team_emb = self.gnn_forward(g)
        return self._heads(emp_emb, demand_emb, team_emb, demand_ids, team_ids,
                           slot_gaps, shift, kind, action_masks)


In [4]:
from dataclasses import dataclass, field
from torch.distributions import Categorical


@dataclass
class Trajectory:
    demand_ids: list = field(default_factory=list)   # slot-queue index = demand node id
    team_ids: list = field(default_factory=list)
    slot_gaps: list = field(default_factory=list)
    shifts: list = field(default_factory=list)
    kinds: list = field(default_factory=list)
    action_masks: list = field(default_factory=list)
    actions: list = field(default_factory=list)
    log_probs_old: list = field(default_factory=list)
    rewards: list = field(default_factory=list)
    values: list = field(default_factory=list)
    snap_ids: list = field(default_factory=list)
    snap_feats: list = field(default_factory=list)   # per-snapshot node-feature dicts (CPU);
                                                     # topology is static, so no graphs stored

    def to_tensors(self):
        return {
            "demand_ids": torch.tensor(self.demand_ids, dtype=torch.long),
            "team_ids": torch.tensor(self.team_ids, dtype=torch.long),
            "slot_gaps": torch.stack(self.slot_gaps),
            "shifts": torch.stack(self.shifts),
            "kinds": torch.stack(self.kinds),
            "action_masks": torch.stack(self.action_masks),
            "actions": torch.tensor(self.actions, dtype=torch.long),
            "log_probs_old": torch.tensor(self.log_probs_old, dtype=torch.float32),
            "rewards": torch.tensor(self.rewards, dtype=torch.float32),
            "values": torch.tensor(self.values, dtype=torch.float32),
            "snap_ids": torch.tensor(self.snap_ids, dtype=torch.long),
            "snap_feats": self.snap_feats,   # list of {ntype: tensor}
        }


def collect_trajectory(env, model):
    model.eval()
    traj = Trajectory()
    env.reset()
    graph = get_graph(env)
    terminated = truncated = False

    cached_day_id = None
    cached_emp_emb = None
    cached_demand_emb = None
    cached_team_emb = None

    with torch.no_grad():
        while not (terminated or truncated):
            slot = env.current_slot()
            if slot is None:
                break
            day_id, s_idx, t_idx, kind = slot
            slot_id = env.slot_idx   # demand node id of the slot being decided

            if day_id != cached_day_id:
                feats = update_graph_features(graph, env)
                cached_emp_emb, cached_demand_emb, cached_team_emb = model.gnn_forward(graph)
                cached_day_id = day_id
                traj.snap_feats.append(feats)
                cur_snap = len(traj.snap_feats) - 1

            # Per-step inputs describing THIS slot.
            gap = slot_gap(env, day_id, s_idx, t_idx)
            shift_oh = [0.0] * env.num_shifts; shift_oh[s_idx] = 1.0
            mask = env.get_employee_mask()

            demand_id_t = torch.tensor([slot_id], dtype=torch.long)
            team_id_t = torch.tensor([t_idx], dtype=torch.long)
            gap_t = torch.from_numpy(gap).unsqueeze(0)
            shift_t = torch.tensor([shift_oh], dtype=torch.float32)
            kind_t = torch.tensor([[float(kind)]], dtype=torch.float32)
            mask_t = torch.tensor(mask.tolist(), dtype=torch.bool).unsqueeze(0)

            probs, values = model._heads(
                cached_emp_emb, cached_demand_emb, cached_team_emb,
                demand_id_t, team_id_t, gap_t,
                shift_t, kind_t, mask_t,
            )

            dist = Categorical(probs=probs[0])
            action = dist.sample()

            _, reward, terminated, truncated, _ = env.step(action.item())

            traj.demand_ids.append(slot_id)
            traj.team_ids.append(t_idx)
            traj.slot_gaps.append(gap_t[0])
            traj.shifts.append(shift_t[0])
            traj.kinds.append(kind_t[0])
            traj.action_masks.append(mask_t[0])
            traj.actions.append(action.item())
            traj.log_probs_old.append(dist.log_prob(action).item())
            traj.rewards.append(float(reward))
            traj.values.append(values[0].item())
            traj.snap_ids.append(cur_snap)

    return traj


In [5]:
def compute_gae(rewards, values, gamma=0.99, lam=0.95):
    T = len(rewards)
    rewards_a = np.array(rewards, dtype=np.float32)
    values_a = np.array(values,  dtype=np.float32)
    values_ext = np.append(values_a, 0.0)

    advantages = np.zeros(T, dtype=np.float32)
    gae = 0.0
    for t in reversed(range(T)):
        delta = rewards_a[t] + gamma * values_ext[t + 1] - values_ext[t]
        gae = delta + gamma * lam * gae
        advantages[t] = gae

    returns = advantages + values_a
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-5)

    return (
        torch.tensor(advantages, dtype=torch.float32),
        torch.tensor(returns, dtype=torch.float32),
    )


In [6]:
from torch.distributions import Categorical


def ppo_update_multi(model, optimizer, batches,
                     clip_eps=0.2, value_coeff=0.5, entropy_coeff=0.01,
                     K_epochs=4, mini_batch_size=512, target_kl=0.02):
    # batches: list of (static_graph, batch, advantages, returns), one per
    # scenario. Interleaving is unchanged from v3 (round-robin minibatches so
    # the shared heads never get a long single-task stretch). What changed for
    # the Option 7 graph: topology is static per scenario, so a snapshot is
    # just a node-feature dict, and a minibatch's graph is n batched copies of
    # the scenario's one graph with those features loaded.
    streams = []
    for graph, batch, advantages, returns in batches:
        snap_ids = batch["snap_ids"]
        dev_batch = {k: v for k, v in batch.items() if torch.is_tensor(v)}
        streams.append({
            "batch": dev_batch, "adv": advantages, "ret": returns,
            "snap_ids": snap_ids, "feats": batch["snap_feats"], "g": graph,
            "T": batch["actions"].shape[0],
            "U": int(snap_ids.max().item()) + 1,
            "E": graph.num_nodes("employee"),
            "ND": graph.num_nodes("demand"),
            "TM": graph.num_nodes("team"),
            "bg_cache": {},   # n -> batched skeleton; features are overwritten per minibatch
        })

    losses, track_entropy, track_kl = [], [], []
    model.train()

    for _ in range(K_epochs):
        # Per stream: shuffle whole snapshot groups (keeps each day's steps
        # together so a minibatch touches only a handful of snapshots), then
        # interleave the streams' minibatches round-robin.
        schedules = []
        for s in streams:
            rank = torch.randperm(s["U"])
            order = torch.argsort(rank[s["snap_ids"]], stable=True)
            schedules.append([order[i:i + mini_batch_size]
                              for i in range(0, s["T"], mini_batch_size)])
        interleaved = []
        for k in range(max(len(mbs) for mbs in schedules)):
            for si, mbs in enumerate(schedules):
                if k < len(mbs):
                    interleaved.append((si, mbs[k]))

        epoch_kls = []
        for si, idx in interleaved:
            s = streams[si]
            batch, advantages, returns = s["batch"], s["adv"], s["ret"]
            snaps, local = torch.unique(s["snap_ids"][idx], return_inverse=True)
            n = snaps.shape[0]

            bg = s["bg_cache"].get(n)
            if bg is None:
                bg = dgl.batch([s["g"]] * n)
                s["bg_cache"][n] = bg
            for ntype in ("employee", "demand", "team"):
                bg.nodes[ntype].data["feat"] = torch.cat(
                    [s["feats"][i][ntype] for i in snaps.tolist()])

            emp_emb, demand_emb, team_emb = model.gnn_forward(bg)
            emp_emb = emp_emb.view(n, s["E"], -1)
            demand_emb = demand_emb.view(n, s["ND"], -1)
            team_emb = team_emb.view(n, s["TM"], -1)

            idx_d = idx
            local_d = local
            probs, values = model._heads_multi(
                emp_emb[local_d],
                demand_emb[local_d, batch["demand_ids"][idx_d]],
                team_emb[local_d, batch["team_ids"][idx_d]],
                batch["slot_gaps"][idx_d],
                batch["shifts"][idx_d], batch["kinds"][idx_d],
                batch["action_masks"][idx_d],
            )

            dist = Categorical(probs=probs)
            logp = dist.log_prob(batch["actions"][idx_d])
            ratios = torch.exp(logp - batch["log_probs_old"][idx_d])
            with torch.no_grad():
                # Schulman's non-negative approx-KL between the policy that
                # collected the data and the current one.
                approx_kl = ((ratios - 1) - torch.log(ratios)).mean().item()
            epoch_kls.append(approx_kl)
            adv = advantages[idx_d]
            surr1 = ratios * adv
            surr2 = torch.clamp(ratios, 1 - clip_eps, 1 + clip_eps) * adv
            actor_loss = -torch.min(surr1, surr2).mean() - entropy_coeff * dist.entropy().mean()
            critic_loss = F.smooth_l1_loss(values, returns[idx_d])
            loss = actor_loss + value_coeff * critic_loss

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            optimizer.step()
            losses.append(loss.item())
            track_entropy.append(dist.entropy().mean().item())

        track_kl.extend(epoch_kls)
        # KL guard: once an epoch has moved the policy too far from the one
        # that collected the data, further epochs on the same trajectories
        # are destructive — stop early (standard PPO target_kl brake).
        if target_kl is not None and float(np.mean(epoch_kls)) > target_kl:
            break

    return float(np.mean(losses)), float(np.mean(track_entropy)), float(np.mean(track_kl))


In [7]:
import os

# Same notebook runs locally (smoke test) and on Kaggle (training).
ON_KAGGLE = os.path.exists("/kaggle/input")
if ON_KAGGLE:
    PROBLEMS_ROOT = "/kaggle/input/datasets/monteiroo21"
    POOL = ["2teams", "4teams"]   # training scenarios
    HOLDOUT = "8teams"            # never trained on: zero-shot eval
    WORK_DIR = "/kaggle/working"
else:
    PROBLEMS_ROOT = "../../../../data/problems"
    POOL = ["SMARTASK_SIMPLE_2025", "SMARTASK_4TEAMS_24EMP"]
    HOLDOUT = "SMARTASK_8TEAMS_48EMP"
    WORK_DIR = "."

_scenario_cache = {}

def get_scenario(name):
    if name not in _scenario_cache:
        _scenario_cache[name] = ScheduleEnv(data_dir=f"{PROBLEMS_ROOT}/{name}")
    return _scenario_cache[name]


def evaluate(model, env, greedy=False):
    model.eval()
    env.reset()
    graph = get_graph(env)
    terminated = truncated = False
    total_reward = 0.0

    cached_day_id = None
    cached_emp_emb = None
    cached_demand_emb = None
    cached_team_emb = None

    with torch.no_grad():
        while not (terminated or truncated):
            slot = env.current_slot()
            if slot is None:
                break
            day_id, s_idx, t_idx, kind = slot
            slot_id = env.slot_idx

            if day_id != cached_day_id:
                update_graph_features(graph, env)
                cached_emp_emb, cached_demand_emb, cached_team_emb = model.gnn_forward(graph)
                cached_day_id = day_id

            gap = slot_gap(env, day_id, s_idx, t_idx)
            shift_oh = [0.0] * env.num_shifts; shift_oh[s_idx] = 1.0
            mask = env.get_employee_mask()

            demand_id_t = torch.tensor([slot_id], dtype=torch.long)
            team_id_t = torch.tensor([t_idx], dtype=torch.long)
            gap_t = torch.from_numpy(gap).unsqueeze(0)
            shift_t = torch.tensor([shift_oh], dtype=torch.float32)
            kind_t = torch.tensor([[float(kind)]], dtype=torch.float32)
            mask_t = torch.tensor(mask.tolist(), dtype=torch.bool).unsqueeze(0)

            probs, _ = model._heads(
                cached_emp_emb, cached_demand_emb, cached_team_emb,
                demand_id_t, team_id_t, gap_t, shift_t, kind_t, mask_t,
            )

            action = probs[0].argmax() if greedy else Categorical(probs=probs[0]).sample()
            _, reward, terminated, truncated, _ = env.step(action.item())
            total_reward += reward

    shortfall = int(np.maximum(0, env.min_demand - env.daily_coverage).sum())
    ideal_gap = int(np.maximum(0, env.ideal_demand - env.daily_coverage).sum())

    snapshot = {
        "demand_skips": env.demand_skips,
        "ideal_gap": ideal_gap,
        "daily_coverage": env.daily_coverage.copy(),
    }

    schedule = [
        [env.action_label(emp, day) for day in range(env.num_days)]
        for emp in range(env.num_employees)
    ]

    return total_reward, shortfall, env.days_worked.tolist(), schedule, snapshot


In [8]:
# --- Smoke test --------------------------------------------------------------
# Cheap functional gate before the long Kaggle run:
#   1. static-graph construction + demand-node <-> slot-queue invariant
#   2. one full collected episode per pool scenario + one joint PPO update
#   3. checkpoint save/load round-trip
# Runs automatically when local; on Kaggle it is skipped (set SMOKE = True to force).
SMOKE = not ON_KAGGLE

if SMOKE:
    import time

    smoke_model = GNNActorCritic.from_env(get_scenario(POOL[0]))

    for name in POOL + [HOLDOUT]:
        env_s = get_scenario(name)
        env_s.reset()
        g = get_graph(env_s)
        update_graph_features(g, env_s)
        ND = g.num_nodes("demand")
        assert ND == len(env_s.slot_queue)
        S = env_s.num_shifts
        c = _o7_static(env_s)
        for i in np.random.default_rng(0).integers(0, ND, 100):
            d, s, t, k = env_s.slot_queue[int(i)]
            assert c["slot_day"][i] == d and c["slot_shift"][i] == s and c["slot_team"][i] == t
            feats = g.nodes["demand"].data["feat"][int(i)]
            assert feats[s].item() == 1.0 and feats[S].item() == float(k)
        deg = g.in_degrees(etype="qualified").cpu().numpy()
        print(f"{name}: {ND} demand nodes | {g.num_edges('qualified')} qualified edges | "
              f"{(deg == 0).sum()} demand nodes with zero qualified employees")

    t0 = time.time()
    smoke_batches = []
    for name in POOL:
        env_s = get_scenario(name)
        traj = collect_trajectory(env_s, smoke_model)
        batch = traj.to_tensors()
        adv, ret = compute_gae(traj.rewards, traj.values, 1.0, 0.95)
        smoke_batches.append((get_graph(env_s), batch, adv, ret))
        print(f"{name}: episode of {len(traj.actions)} steps, {len(traj.snap_feats)} snapshots "
              f"({time.time() - t0:.1f}s cumulative)")

    smoke_opt = torch.optim.Adam(smoke_model.parameters(), lr=1e-4)
    loss, ent, kl = ppo_update_multi(smoke_model, smoke_opt, smoke_batches, K_epochs=1)
    print(f"joint update ok: loss={loss:.4f} ent={ent:.4f} kl={kl:.4f} "
          f"({time.time() - t0:.1f}s cumulative)")

    torch.save({"model_state_dict": smoke_model.state_dict()}, f"{WORK_DIR}/smoke_ckpt.pth")
    reload_model = GNNActorCritic.from_env(get_scenario(POOL[0]))
    reload_model.load_state_dict(
        torch.load(f"{WORK_DIR}/smoke_ckpt.pth", weights_only=True)["model_state_dict"])
    os.remove(f"{WORK_DIR}/smoke_ckpt.pth")
    print("checkpoint round-trip ok")

    del smoke_model, smoke_opt, smoke_batches, reload_model


In [9]:
import os

# Hyperparameters — identical to the v3 run except where noted, so the
# Option 7 graph is the only experimental variable.
NUM_ITERATIONS = 5000   # one iteration = one episode from every pool scenario + one joint update
GAMMA = 1.0   # finite-horizon episodic task: terminal reward must reach every decision undiscounted
LAM = 0.95
CLIP_EPS = 0.15
VALUE_COEFF = 0.5
ENTROPY_COEFF_0 = 0.01
ENTROPY_DECAY = 0.999   # per iteration
ENTROPY_MIN = 0.005
TARGET_KL = 0.02        # per-update KL brake: skip remaining epochs when exceeded
K_EPOCHS = 3
MINI_BATCH_SIZE = 512
LR = 1e-4
SAVE_EVERY = 10         # iterations
EVAL_SAMPLES = 5        # sampled rollouts per pool scenario for checkpoint selection
HOLDOUT_EVERY = 50      # iterations, log-only
HOLDOUT_SAMPLES = 2

BEST_CKPT = f"{WORK_DIR}/best_multiteam_option7.pth"
LATEST_CKPT = f"{WORK_DIR}/latest_multiteam_option7.pth"
RESUME_FROM = f"{WORK_DIR}/latest_multiteam_option7.pth"

model = GNNActorCritic.from_env(get_scenario(POOL[0]))
optimizer = torch.optim.Adam(model.parameters(), lr=LR, eps=1e-5)

INF_METRIC = (float("inf"), float("inf"))
best_metric = INF_METRIC
start_iteration = 0

if RESUME_FROM and os.path.exists(RESUME_FROM):
    ckpt = torch.load(RESUME_FROM, weights_only=True)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    for group in optimizer.param_groups:
        group["lr"] = LR
    start_iteration = ckpt["iteration"]
    bm = ckpt.get("best_metric", INF_METRIC)
    best_metric = (float(bm), float("inf")) if isinstance(bm, float) else tuple(bm)
    print(f"Resumed from iteration {start_iteration}, best metric = {best_metric}")


def eval_scenario(model, env):
    # Mean over the eval samples. v3 used best-of-3, which let one scenario's
    # sampling variance mask the other scenario's regressions during
    # checkpoint selection; the mean rewards balanced policies instead.
    sfs, exs = [], []
    for k in range(EVAL_SAMPLES):
        torch.manual_seed(1234 + k)
        _, sf, _, _, snap = evaluate(model, env, greedy=False)
        sfs.append(sf)
        exs.append(max(0, snap["ideal_gap"] - env.ideal_floor))
    return float(np.mean(sfs)), float(np.mean(exs))


for iteration in range(start_iteration, NUM_ITERATIONS):
    entropy_coeff = max(ENTROPY_MIN, ENTROPY_COEFF_0 * (ENTROPY_DECAY ** iteration))

    # Collect one episode per pool scenario; the update sees them jointly.
    # Advantages are normalized per scenario inside compute_gae, so neither
    # scenario's return scale dominates the shared heads.
    batches, ep_stats = [], []
    for name in POOL:
        env = get_scenario(name)
        traj = collect_trajectory(env, model)
        batch = traj.to_tensors()
        advantages, returns = compute_gae(traj.rewards, traj.values, GAMMA, LAM)
        batches.append((get_graph(env), batch, advantages, returns))

        ideal_gap = int(np.maximum(0, env.ideal_demand - env.daily_coverage).sum())
        ep_stats.append({
            "name": name,
            "R": sum(traj.rewards),
            "sf": int(np.maximum(0, env.min_demand - env.daily_coverage).sum()),
            "ex": max(0, ideal_gap - env.ideal_floor),
            "ds": int(np.maximum(0, env.max_days_per_year - env.days_worked).sum()),
        })

    mean_loss, mean_entropy, mean_kl = ppo_update_multi(
        model, optimizer, batches,
        clip_eps=CLIP_EPS,
        value_coeff=VALUE_COEFF,
        entropy_coeff=entropy_coeff,
        K_epochs=K_EPOCHS,
        mini_batch_size=MINI_BATCH_SIZE,
        target_kl=TARGET_KL,
    )

    if iteration % 5 == 0:
        parts = " | ".join(
            f"[{s['name']}] R={s['R']:.0f} sf={s['sf']} ex={s['ex']} days_short={s['ds']}"
            for s in ep_stats)
        print(f"It {iteration:>4} | Loss={mean_loss:.4f} | ent={mean_entropy:.4f} "
              f"(coeff={entropy_coeff:.4f}) | kl={mean_kl:.4f} | " + parts)

    if (iteration + 1) % SAVE_EVERY == 0:
        rng_state = torch.get_rng_state()
        metric_sf, metric_ex = 0.0, 0.0
        parts = []
        for eval_name in POOL:
            eval_env = get_scenario(eval_name)
            sf, ex = eval_scenario(model, eval_env)
            total_min = eval_env.min_demand.sum()
            metric_sf += sf / total_min
            metric_ex += ex / total_min
            parts.append(f"{eval_name}: shortfall={sf:.1f} "
                         f"({100 * (1 - sf / total_min):.2f}% cov) excess={ex:.1f}")
        torch.set_rng_state(rng_state)
        metric = (float(metric_sf), float(metric_ex))

        print(f"  eval (mean of {EVAL_SAMPLES}) @ it {iteration + 1}: " + " | ".join(parts))
        if metric < best_metric:
            best_metric = metric
            torch.save(
                {"iteration": iteration + 1, "model_state_dict": model.state_dict(),
                 "optimizer_state_dict": optimizer.state_dict(), "best_metric": list(best_metric)},
                BEST_CKPT,
            )
            print(f"  NEW BEST: shortfall frac = {metric_sf:.4f} | excess frac = {metric_ex:.4f}")

        # LATEST is saved after the eval so a resume picks up the current
        # best_metric.
        torch.save(
            {"iteration": iteration + 1, "model_state_dict": model.state_dict(),
             "optimizer_state_dict": optimizer.state_dict(), "best_metric": list(best_metric)},
            LATEST_CKPT,
        )

    if (iteration + 1) % HOLDOUT_EVERY == 0:
        rng_state = torch.get_rng_state()
        h_env = get_scenario(HOLDOUT)
        h_sfs, h_exs = [], []
        for k in range(HOLDOUT_SAMPLES):
            torch.manual_seed(1234 + k)
            _, sf, _, _, snap = evaluate(model, h_env, greedy=False)
            h_sfs.append(sf)
            h_exs.append(max(0, snap["ideal_gap"] - h_env.ideal_floor))
        torch.set_rng_state(rng_state)
        print(f"  [holdout {HOLDOUT}] @ it {iteration + 1}: "
              f"shortfall={np.mean(h_sfs):.1f} | ideal_excess={np.mean(h_exs):.1f} (log-only)")


Resumed from iteration 60, best metric = (0.00660200422910729, 0.17709439314937153)
It   60 | Loss=5.6539 | ent=0.7581 (coeff=0.0094) | kl=0.0012 | [2teams] R=2367 sf=15 ex=229 days_short=0 | [4teams] R=5320 sf=10 ex=159 days_short=0
It   65 | Loss=5.6709 | ent=0.7788 (coeff=0.0094) | kl=0.0004 | [2teams] R=2204 sf=10 ex=241 days_short=44 | [4teams] R=5281 sf=12 ex=184 days_short=0
  eval (mean of 5) @ it 70: 2teams: shortfall=12.0 (99.42% cov) excess=255.2 | 4teams: shortfall=3.8 (99.87% cov) excess=184.4
It   70 | Loss=5.8611 | ent=0.8006 (coeff=0.0093) | kl=0.0011 | [2teams] R=1906 sf=11 ex=258 days_short=113 | [4teams] R=5316 sf=3 ex=193 days_short=0
It   75 | Loss=5.8914 | ent=0.7853 (coeff=0.0093) | kl=0.0005 | [2teams] R=1792 sf=15 ex=263 days_short=135 | [4teams] R=5316 sf=5 ex=184 days_short=0
  eval (mean of 5) @ it 80: 2teams: shortfall=14.2 (99.32% cov) excess=247.2 | 4teams: shortfall=5.2 (99.82% cov) excess=182.0
It   80 | Loss=5.8669 | ent=0.7860 (coeff=0.0092) | kl=0.00

KeyboardInterrupt: 

In [10]:
import csv

ckpt = torch.load(BEST_CKPT, weights_only=True)
model.load_state_dict(ckpt["model_state_dict"])
bm = ckpt["best_metric"]
bm = bm if isinstance(bm, float) else tuple(bm)   # metric is (shortfall frac, excess frac)
ckpt_step = ckpt.get("iteration", ckpt.get("episode"))
print(f"Checkpoint: iteration {ckpt_step}, best metric = {bm}\n")

N = 20
for name in POOL + [HOLDOUT]:
    env_e = get_scenario(name)
    total_min = env_e.min_demand.sum()
    tag = "ZERO-SHOT (never trained on)" if name == HOLDOUT else "trained"
    print(f"{'=' * 70}")
    print(f"{name} ({tag}) — {env_e.num_teams} teams, {env_e.num_employees} employees, "
          f"total min demand {total_min}, ideal floor {env_e.ideal_floor}")
    print(f"{'=' * 70}")

    results = []
    for i in range(N):
        torch.manual_seed(42 + i)  # sampling is torch-based, so seed torch (not numpy)
        reward, shortfall, days_worked, schedule, snap = evaluate(model, env_e, greedy=False)
        results.append((reward, shortfall, days_worked, schedule, snap))
        print(f"Run {i + 1:2d} | Reward: {reward:8.1f} | Shortfall: {shortfall:4d} "
              f"| ideal_gap: {snap['ideal_gap']:4d} (excess={snap['ideal_gap'] - env_e.ideal_floor:4d}) "
              f"| demand_skips: {snap['demand_skips']:4d} "
              f"| avg days={np.mean(days_worked):.0f}")

    # Mean over the fixed-seed runs is the honest statistic; best-of-N is the
    # deployment strategy (generate N schedules, keep the best).
    mean_sf = np.mean([r[1] for r in results])
    mean_ex = np.mean([r[4]["ideal_gap"] - env_e.ideal_floor for r in results])
    print(f"\nMEAN over {N} runs | Shortfall: {mean_sf:.1f} | Ideal excess: {mean_ex:.1f}")

    # Pick the run with the lowest shortfall, breaking ties on ideal_gap
    best_idx = min(range(N), key=lambda i: (results[i][1], results[i][4]["ideal_gap"]))
    best_reward, best_shortfall, best_days, best_schedule, best_snap = results[best_idx]

    print(f"BEST: Run {best_idx + 1} | Shortfall: {best_shortfall} "
          f"({100 * (1 - best_shortfall / total_min):.2f}% coverage) "
          f"| Ideal gap: {best_snap['ideal_gap']} "
          f"(excess={best_snap['ideal_gap'] - env_e.ideal_floor})")
    print(f"Days worked/employee: avg={np.mean(best_days):.0f}, "
          f"min={min(best_days)}, max={max(best_days)}")

    # Per-day shortfall breakdown for the best run, naming every (shift, team) gap.
    best_coverage = best_snap["daily_coverage"]
    for d in range(env_e.num_days):
        gaps = []
        for s_idx, shift in enumerate(env_e.shift_codes):
            for t_idx, team in enumerate(env_e.teams):
                gap = int(env_e.min_demand[d, s_idx, t_idx] - best_coverage[d, s_idx, t_idx])
                if gap > 0:
                    gaps.append(f"{shift}-{team}={gap}")
        if gaps:
            print(f"Day {d:3d}: {', '.join(gaps)}")

    schedule_csv = f"{WORK_DIR}/best_schedule_{name}_multiteam_option7.csv"
    with open(schedule_csv, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["Employee"] + [f"Day_{d}" for d in range(len(best_schedule[0]))])
        for emp in range(len(best_schedule)):
            writer.writerow([f"Employee_{emp + 1}"] + best_schedule[emp])
    print(f"\nBest schedule written to {schedule_csv}\n")


Checkpoint: iteration 30, best metric = (0.00660200422910729, 0.17709439314937153)

2teams (trained) — 2 teams, 12 employees, total min demand 2086, ideal floor 0
Run  1 | Reward:   2377.4 | Shortfall:   11 | ideal_gap:  234 (excess= 234) | demand_skips:   11 | avg days=223
Run  2 | Reward:   2390.3 | Shortfall:   11 | ideal_gap:  227 (excess= 227) | demand_skips:   11 | avg days=223
Run  3 | Reward:   2370.0 | Shortfall:   16 | ideal_gap:  223 (excess= 223) | demand_skips:   16 | avg days=223
Run  4 | Reward:   2363.2 | Shortfall:   13 | ideal_gap:  231 (excess= 231) | demand_skips:   13 | avg days=223
Run  5 | Reward:   2350.8 | Shortfall:   16 | ideal_gap:  238 (excess= 238) | demand_skips:   16 | avg days=223
Run  6 | Reward:   2367.3 | Shortfall:   11 | ideal_gap:  245 (excess= 245) | demand_skips:   11 | avg days=223
Run  7 | Reward:   2374.9 | Shortfall:   10 | ideal_gap:  240 (excess= 240) | demand_skips:   10 | avg days=223
Run  8 | Reward:   2354.6 | Shortfall:   17 | ideal_g

In [ ]:
import matplotlib.pyplot as plt

PLOT_SCENARIO = HOLDOUT   # any name from POOL, or HOLDOUT for the zero-shot scenario
env_p = get_scenario(PLOT_SCENARIO)

ckpt = torch.load(BEST_CKPT, weights_only=True)
model.load_state_dict(ckpt["model_state_dict"])
_, _, days_worked_list, _, snap = evaluate(model, env_p, greedy=False)

days   = np.arange(env_p.num_days)
cov    = snap["daily_coverage"]   # (num_days, num_shifts, num_teams)
demand = env_p.min_demand         # (num_days, num_shifts, num_teams)

# Coverage-vs-demand grid: one subplot per (shift, team) pair.
fig, axes = plt.subplots(
    env_p.num_shifts, env_p.num_teams,
    figsize=(4 * env_p.num_teams, 4 * env_p.num_shifts),
    sharex=True, sharey=True, squeeze=False,
)
for s_idx, shift in enumerate(env_p.shift_codes):
    for t_idx, team in enumerate(env_p.teams):
        ax = axes[s_idx][t_idx]
        ax.plot(days, cov[:, s_idx, t_idx], label='Coverage', alpha=0.7)
        ax.plot(days, demand[:, s_idx, t_idx], label='Demand', alpha=0.7, linestyle='--', color='r')
        ax.fill_between(
            days, cov[:, s_idx, t_idx], demand[:, s_idx, t_idx],
            where=cov[:, s_idx, t_idx] < demand[:, s_idx, t_idx],
            color='red', alpha=0.2, label='Shortfall',
        )
        ax.set_title(f"{shift}-{team}")
        if t_idx == 0:
            ax.set_ylabel('Employees')
        if s_idx == env_p.num_shifts - 1:
            ax.set_xlabel('Day of year')
        ax.legend(loc='upper right', fontsize=8)

fig.suptitle('Coverage vs Demand across the year', fontsize=14)
plt.tight_layout()
plt.show()

# Quarterly violation breakdown — one column per (shift, team) pair plus a total.
pair_labels = [f"{s}-{t}" for t in env_p.teams for s in env_p.shift_codes]
header = f"{'Quarter':<12} " + " ".join(f"{p:>8}" for p in pair_labels) + f" {'Total':>6}"
print()
print(header)
print("-" * len(header))
total_viol = 0
for q, (start, end) in enumerate([(0, 91), (91, 182), (182, 273), (273, 365)], 1):
    per_pair = []
    q_total = 0
    for team in env_p.teams:
        for shift in env_p.shift_codes:
            s_idx, t_idx = env_p.shift_idx[shift], env_p.team_idx[team]
            v = int(np.sum(np.maximum(0, demand[start:end, s_idx, t_idx] - cov[start:end, s_idx, t_idx])))
            per_pair.append(v)
            q_total += v
    total_viol += q_total
    print(f"Q{q} ({start:3d}-{end:3d})  " + " ".join(f"{v:>8}" for v in per_pair) + f" {q_total:>6}")
print("-" * len(header))
print(f"{'Total':<12} " + " ".join(f"{'':>8}" for _ in pair_labels) + f" {total_viol:>6}")

days_worked_arr = np.array(days_worked_list)
print(f"\nDays worked/employee: {days_worked_list}")
print(f"Mean: {days_worked_arr.mean():.1f}, Min: {days_worked_arr.min()}, Max: {days_worked_arr.max()}")
